In [ ]:
# Environment setup and Colab detection
# Ref: setup_colab.ipynb for Colab integration patterns
import sys
import os
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🌐 Running in Google Colab")
    from google.colab import drive
    drive.mount('/content/drive')

    # Set up paths for Colab - following setup_colab.ipynb patterns
    DRIVE_ROOT = '/content/drive/MyDrive'
    PROJECT_ROOT = f'{DRIVE_ROOT}/RL-CC-SAM'

    # Change to project directory
    os.chdir(PROJECT_ROOT)
    sys.path.append(PROJECT_ROOT)

    print(f"📁 Working directory: {os.getcwd()}")
else:
    print("💻 Running locally")
    # Assume notebook is in notebooks/ folder
    PROJECT_ROOT = Path.cwd().parent
    os.chdir(PROJECT_ROOT)

    print(f"📁 Working directory: {PROJECT_ROOT}")

# Set up directories following project structure
DATASETS_DIR = Path(PROJECT_ROOT) / "datasets"
PRETRAINED_DIR = Path(PROJECT_ROOT) / "pretrained"
SEGMENT_ANYTHING_DIR = Path(PROJECT_ROOT) / "segment-anything"

# Create directories
for dir_path in [DATASETS_DIR, PRETRAINED_DIR]:
    dir_path.mkdir(exist_ok=True)

print(f"📊 Datasets directory: {DATASETS_DIR}")
print(f"🧠 Pretrained models directory: {PRETRAINED_DIR}")
print(f"🔧 Segment Anything submodule: {SEGMENT_ANYTHING_DIR}")

# Verify submodules are available (following .cursor/rules/submodule-handling.mdc)
if not SEGMENT_ANYTHING_DIR.exists():
    print("❌ Segment Anything submodule not found!")
    print("💡 Run: git submodule update --init --recursive")
    print("💡 Or use setup_colab.ipynb for initial setup")
else:
    print("✅ Segment Anything submodule found")

print("✅ Environment setup complete")


In [ ]:
# Import essential libraries
# All dependencies should be installed via requirements.txt

# Core libraries
import torch
import numpy as np
import matplotlib.pyplot as plt
import cv2
import json
import math
import glob
from tqdm.auto import tqdm

# Medical imaging
import nibabel as nib
from scipy.ndimage import distance_transform_edt

# SAM - using existing submodule (following .cursor/rules/submodule-handling.mdc)
sys.path.insert(0, str(SEGMENT_ANYTHING_DIR))
from segment_anything import sam_model_registry, SamPredictor

# Image processing
from skimage import io, transform
import torch.nn.functional as F

# Device handling following .cursor/rules/pytorch-devices.mdc
def get_device():
    """Get optimal device following PyTorch device handling rules."""
    if torch.backends.mps.is_available():
        return torch.device("mps")  # Apple Silicon
    elif torch.cuda.is_available():
        return torch.device("cuda")  # NVIDIA GPU
    else:
        return torch.device("cpu")   # CPU fallback

device = get_device()
print(f"🚀 Using device: {device}")

# Set style for plots
plt.style.use('seaborn-v0_8' if 'seaborn-v0_8' in plt.style.available else 'default')
print("✅ Libraries imported successfully")


In [ ]:
# Define comprehensive evaluation metrics for medical image segmentation
def compute_dice(pred_mask: np.ndarray, true_mask: np.ndarray) -> float:
    """
    Calculate Dice Coefficient (F1 score) between two binary masks.

    Args:
        pred_mask: Predicted binary mask
        true_mask: Ground truth binary mask

    Returns:
        float: Dice coefficient [0, 1], where 1 is perfect overlap
    """
    intersection = np.logical_and(pred_mask, true_mask).sum()
    size_pred = pred_mask.sum()
    size_true = true_mask.sum()

    if size_pred + size_true == 0:
        return 1.0  # Both masks are empty - perfect agreement

    return 2.0 * intersection / (size_pred + size_true + 1e-8)

def compute_iou(pred_mask: np.ndarray, true_mask: np.ndarray) -> float:
    """
    Calculate Intersection over Union (IoU) between two binary masks.

    Args:
        pred_mask: Predicted binary mask
        true_mask: Ground truth binary mask

    Returns:
        float: IoU score [0, 1], where 1 is perfect overlap
    """
    intersection = np.logical_and(pred_mask, true_mask).sum()
    union = np.logical_or(pred_mask, true_mask).sum()

    if union == 0:
        return 1.0  # Both masks are empty

    return intersection / (union + 1e-8)

def compute_hausdorff(pred_mask: np.ndarray, true_mask: np.ndarray) -> float:
    """
    Calculate Hausdorff distance between two binary masks.

    Args:
        pred_mask: Predicted binary mask
        true_mask: Ground truth binary mask

    Returns:
        float: Hausdorff distance in pixels (lower is better)
    """
    if pred_mask.sum() == 0 or true_mask.sum() == 0:
        return math.inf  # Cannot compute distance if either mask is empty

    # Calculate distance transforms
    dt_true = distance_transform_edt(~true_mask.astype(bool))
    dt_pred = distance_transform_edt(~pred_mask.astype(bool))

    # Calculate directed Hausdorff distances
    hd1 = dt_pred[true_mask.astype(bool)].max()  # GT boundary to pred region
    hd2 = dt_true[pred_mask.astype(bool)].max()  # Pred boundary to GT region

    return max(hd1, hd2)

def compute_sensitivity(pred_mask: np.ndarray, true_mask: np.ndarray) -> float:
    """Calculate sensitivity (recall/true positive rate)."""
    tp = np.logical_and(pred_mask, true_mask).sum()
    fn = np.logical_and(~pred_mask, true_mask).sum()

    if tp + fn == 0:
        return 1.0  # No positive cases

    return tp / (tp + fn)

def compute_specificity(pred_mask: np.ndarray, true_mask: np.ndarray) -> float:
    """Calculate specificity (true negative rate)."""
    tn = np.logical_and(~pred_mask, ~true_mask).sum()
    fp = np.logical_and(pred_mask, ~true_mask).sum()

    if tn + fp == 0:
        return 1.0  # No negative cases

    return tn / (tn + fp)

def evaluate_segmentation(pred_mask: np.ndarray, true_mask: np.ndarray) -> dict:
    """
    Comprehensive evaluation of segmentation performance.

    Returns:
        dict: Dictionary containing all evaluation metrics
    """
    return {
        'dice': compute_dice(pred_mask, true_mask),
        'iou': compute_iou(pred_mask, true_mask),
        'hausdorff': compute_hausdorff(pred_mask, true_mask),
        'sensitivity': compute_sensitivity(pred_mask, true_mask),
        'specificity': compute_specificity(pred_mask, true_mask)
    }

# Visualization functions
def show_mask(mask, ax, random_color=False, alpha=0.6):
    """Display segmentation mask overlay."""
    if random_color:
        color = np.concatenate([np.random.random(3), np.array([alpha])], axis=0)
    else:
        color = np.array([251/255, 252/255, 30/255, alpha])  # Yellow

    h, w = mask.shape[-2:]
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    ax.imshow(mask_image)

def show_box(box, ax, color='blue'):
    """Draw bounding box on matplotlib axis."""
    x0, y0 = box[0], box[1]
    w, h = box[2] - box[0], box[3] - box[1]
    ax.add_patch(plt.Rectangle((x0, y0), w, h, edgecolor=color, facecolor=(0,0,0,0), lw=2))

print("✅ Evaluation metrics and helper functions defined")
print("📊 Available metrics: Dice, IoU, Hausdorff Distance, Sensitivity, Specificity")


In [ ]:
# Load SAM model (assumes already downloaded via setup_colab.ipynb)
# Following .cursor/rules: Downloads handled in setup_colab.ipynb

def load_sam_model():
    """
    Load SAM model from pretrained directory.

    Prerequisites:
    - Run setup_colab.ipynb first to download the model
    - Model should be at: pretrained/sam_vit_b_01ec64.pth or similar
    """
    
    # Try different SAM model variants
    sam_model_files = [
        "sam_vit_b_01ec64.pth",  # ViT-B model
        "sam_vit_l_0b3195.pth",  # ViT-L model  
        "sam_vit_h_4b8939.pth",  # ViT-H model
    ]
    
    model_path = None
    model_type = None
    
    # Find available SAM model
    for filename in sam_model_files:
        potential_path = PRETRAINED_DIR / filename
        if potential_path.exists():
            model_path = potential_path
            if "vit_b" in filename:
                model_type = "vit_b"
            elif "vit_l" in filename:
                model_type = "vit_l"
            elif "vit_h" in filename:
                model_type = "vit_h"
            break
    
    # Check if model exists
    if model_path is None:
        print(f"❌ SAM model not found in: {PRETRAINED_DIR}")
        print("💡 Please run setup_colab.ipynb first to download the model")
        print("💡 The setup notebook will download all required models and datasets")
        print("💡 Available models: sam_vit_b_01ec64.pth, sam_vit_l_0b3195.pth, sam_vit_h_4b8939.pth")
        raise FileNotFoundError(f"SAM model not found. Run setup_colab.ipynb first.")

    try:
        print(f"🔄 Loading SAM model from: {model_path}")
        print(f"📱 Model type: {model_type}")

        # Load using SAM architecture
        sam_model = sam_model_registry[model_type](checkpoint=str(model_path))
        sam_model = sam_model.to(device)
        sam_model.eval()

        # Create predictor
        predictor = SamPredictor(sam_model)

        print(f"✅ SAM model loaded successfully")
        print(f"   Model size: {model_path.stat().st_size / (1024**3):.1f} GB")
        print(f"   Model parameters: {sum(p.numel() for p in sam_model.parameters()):,}")
        print(f"   Device: {device}")

        return sam_model, predictor

    except Exception as e:
        print(f"❌ Error loading SAM model: {e}")
        print("💡 The model file might be corrupted. Re-run setup_colab.ipynb")
        raise

# Load the SAM model
sam_model, predictor = load_sam_model()


In [ ]:
MEDICAL_DECATHLON_DIR = DATASETS_DIR / "medical_decathlon"
Task04_Hippocampus_dir = MEDICAL_DECATHLON_DIR / "Task04_Hippocampus"
Task09_Spleen_dir = MEDICAL_DECATHLON_DIR / "Task09_Spleen"
BUSI_Dataset_dir = DATASETS_DIR / "BUSI_Dataset"


In [ ]:
# Code traverses each slice of MRI volume data, keeping only slices containing hippocampus annotations to reduce unnecessary computation. Each slice is normalized to three-channel 8-bit images (SAM requires RGB image input). mri_slices list will be used for model inference, mri_slice_masks for corresponding ground truth masks.
import nibabel as nib
import numpy as np

# Get Hippocampus training set file list
imagesTr = list(Path(f"{Task04_Hippocampus_dir}/imagesTr").glob("*.nii.gz"))
labelsTr = list(Path(f"{Task04_Hippocampus_dir}/labelsTr").glob("*.nii.gz"))
image_files = sorted([f.name for f in imagesTr])
label_files = sorted([f.name for f in labelsTr])

print(f"Total {len(image_files)} MRI volumes for evaluation.")

# Prepare to store MRI dataset test slices and labels
mri_slices = []      # Will save 2D slice images (numpy arrays)
mri_slice_masks = [] # Will save corresponding 2D GT masks
mri_slice_labels = []# Save mask corresponding anatomical structure labels (1=left hippocampus, 2=right hippocampus)
mri_slice_summaries = []

for img_file, lbl_file in zip(image_files, label_files):
    img_path = f"{Task04_Hippocampus_dir}/imagesTr/{img_file}"
    lbl_path = f"{Task04_Hippocampus_dir}/labelsTr/{lbl_file}"
    # Load NIfTI volume data
    img_nii = nib.load(img_path)
    seg_nii = nib.load(lbl_path)
    img_data = img_nii.get_fdata()
    seg_data = seg_nii.get_fdata()
    volume = img_data.astype(np.float32)  # 3D image data
    seg_volume = seg_data.astype(np.uint8)

    # Traverse each slice (slice direction is axis 2, i.e., axial slices)
    num_slices = img_data.shape[2]
    for z_index in range(num_slices):
        # Take certain axial slice
        slice_img = volume[:, :, z_index]
        slice_mask = seg_volume[:, :, z_index]
        # If this slice has any hippocampus structure annotation, include in evaluation
        if np.any(slice_mask > 0):
            print(f"\rimg_file:'{img_file}', lbl_file:{lbl_file}, |volume|: {volume.shape}, |seg|: {seg_volume.shape}, slice: {z_index}/{num_slices}, Slice shape: {slice_img.shape}, Mask unique labels: {np.unique(slice_mask)}", end='', flush=True)
            mri_slices.append(slice_img)
            mri_slice_masks.append(slice_mask)  # Multi-class labels (values 0,1,2)
            mri_slice_labels.append(lbl_file)  # Record the volume data file this slice belongs to
            mri_slice_summaries.append((z_index, num_slices, slice_mask.sum() / slice_img.shape[0] / slice_img.shape[1]))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
from matplotlib.gridspec import GridSpec
from matplotlib.axes import Axes

def visualize_seg(fig: Figure, axs: Axes, label2box: dict, label2mask: dict, img_rgb: np.ndarray, true_mask: np.ndarray):
    # fig.clear()
    for structure_idx, (label_val, pred_mask) in enumerate(label2mask.items()):
        # Get box for this structure
        input_box = label2box[label_val]
        if input_box is None:
            continue
        x_min, y_min, x_max, y_max = input_box

        row = structure_idx
        # 1. Original image with bounding box
        ax1 = axs[row, 0]
        ax1.clear()
        ax1.imshow(img_rgb[:,:,0], cmap='gray')
        rect = plt.Rectangle((x_min, y_min), x_max-x_min, y_max-y_min,
                            fill=False, edgecolor='red', linewidth=2)
        ax1.add_patch(rect)
        ax1.set_title(f'Input Box - Structure {label_val}')
        ax1.axis('off')

        # 2. True mask for this structure
        ax2 = axs[row, 1]
        ax2.clear()
        structure_true_mask = (true_mask == label_val).astype(np.uint8)
        ax2.imshow(img_rgb[:,:,0], cmap='gray')
        ax2.imshow(np.ma.masked_where(structure_true_mask==0, structure_true_mask),
                  cmap='spring', alpha=0.7)
        ax2.set_title(f'True Mask - Structure {label_val}')
        ax2.axis('off')

        # 3. Predicted mask for this structure
        ax3 = axs[row, 2]
        ax3.clear()
        ax3.imshow(img_rgb[:,:,0], cmap='gray')
        ax3.imshow(np.ma.masked_where(pred_mask==0, pred_mask),
                  cmap='autumn', alpha=0.7)
        ax3.set_title(f'Pred Mask - Structure {label_val}')
        ax3.axis('off')

        # 4. Overlay comparison
        ax4 = axs[row, 3]
        ax4.clear()
        ax4.imshow(img_rgb[:,:,0], cmap='gray')
        # True mask in green, pred mask in red
        ax4.imshow(np.ma.masked_where(structure_true_mask==0, structure_true_mask),
                  cmap='Greens', alpha=0.5)
        ax4.imshow(np.ma.masked_where(pred_mask==0, pred_mask),
                  cmap='Reds', alpha=0.5)
        ax4.set_title(f'Overlay - Structure {label_val}')
        ax4.axis('off')

    # Combined visualization (third row)
    # 1. All bounding boxes
    ax_combined1 = axs[2, 0]
    ax_combined1.clear()
    ax_combined1.imshow(img_rgb[:,:,0], cmap='gray')
    colors = ['red', 'blue']
    for i, (label_val, input_box) in enumerate(label2box.items()):
        x_min, y_min, x_max, y_max = input_box
        rect = plt.Rectangle((x_min, y_min), x_max-x_min, y_max-y_min,
                            fill=False, edgecolor=colors[i % len(colors)], linewidth=2)
        ax_combined1.add_patch(rect)
    ax_combined1.set_title('All Input Boxes')
    ax_combined1.axis('off')

    # 2. Combined true mask
    ax_combined2 = axs[2, 1]
    ax_combined2.clear()
    ax_combined2.imshow(img_rgb[:,:,0], cmap='gray')
    ax_combined2.imshow(np.ma.masked_where((true_mask > 0)==0, (true_mask > 0)),
                        cmap='spring', alpha=0.7)
    ax_combined2.set_title('Combined True Mask')
    ax_combined2.axis('off')

    # 3. Combined predicted mask
    ax_combined3 = axs[2, 2]
    ax_combined3.clear()
    ax_combined3.imshow(img_rgb[:,:,0], cmap='gray')
    ax_combined3.imshow(np.ma.masked_where(pred_mask_total==0, pred_mask_total),
                        cmap='autumn', alpha=0.7)
    ax_combined3.set_title('Combined Pred Mask')
    ax_combined3.axis('off')

    # 4. Final comparison
    ax_combined4 = axs[2, 3]
    ax_combined4.clear()
    ax_combined4.imshow(img_rgb[:,:,0], cmap='gray')
    ax_combined4.imshow(np.ma.masked_where((true_mask > 0)==0, (true_mask > 0)),
                        cmap='Greens', alpha=0.5)
    ax_combined4.imshow(np.ma.masked_where(pred_mask_total==0, pred_mask_total),
                        cmap='Reds', alpha=0.5)
    ax_combined4.set_title('Final Overlay')
    ax_combined4.axis('off')

    fig.canvas.draw()
    fig.canvas.flush_events()

all_labels = [1, 2]

def sam_segment(slice_img: np.ndarray, slice_mask: np.ndarray, predictor: SamPredictor):
    """
    Perform segmentation using SAM predictor on a 2D slice.
    
    Args:
        slice_img: Input image slice
        slice_mask: Ground truth mask  
        predictor: SAM predictor instance
    
    Returns:
        tuple: (label2mask, label2box, pred_mask_total, true_mask, img_rgb)
    """
    # Normalize and prepare image  
    mn, mx = slice_img.min(), slice_img.max()
    slice_img_norm = ((slice_img - mn) / (mx - mn + 1e-8) * 255.0).astype(np.uint8)
    slice_img_resized = cv2.resize(slice_img_norm, (1024, 1024), interpolation=cv2.INTER_CUBIC)
    img_rgb = np.stack([slice_img_resized]*3, axis=-1)

    true_mask = cv2.resize(slice_mask.astype(np.uint8)*127, (1024, 1024), interpolation=cv2.INTER_NEAREST)
    true_mask = true_mask.astype(np.uint8) // 127

    predictor.set_image(img_rgb)  # Set image for SAM predictor
    pred_mask_total = np.zeros(true_mask.shape, dtype=bool)

    # Predict each structure separately (labels 1 and 2 correspond to two hippocampi)
    label2mask = {}
    label2box = {}
    for structure_idx, label_val in enumerate(all_labels):
        # Skip structures that don't exist
        if np.sum(true_mask == label_val) == 0:
            continue

        # Calculate bounding box for this structure
        ys, xs = np.where(true_mask == label_val)
        y_min, y_max = ys.min(), ys.max()
        x_min, x_max = xs.min(), xs.max()
        input_box = np.array([x_min, y_min, x_max, y_max])
        label2box[label_val] = input_box

        # Use bounding box prompt for prediction
        masks, scores, _ = predictor.predict(box=input_box[None, :], point_coords=None, point_labels=None, multimask_output=False)
        pred_mask = masks[0]  # Output mask
        label2mask[label_val] = pred_mask

        # Add this structure's predicted mask to total mask
        pred_mask_total = np.logical_or(pred_mask_total, pred_mask)

    return label2mask, label2box, pred_mask_total, true_mask, img_rgb


In [ ]:
dice_list_hc = []
iou_list_hc = []
hd_list_hc = []

good_slices = []

for idx, (slice_img, slice_mask, slice_label, slice_info) in enumerate(zip(mri_slices, mri_slice_masks, mri_slice_labels, mri_slice_summaries)):
    # if idx > 20:
    #    break
    label2mask, label2box, pred_mask_total, true_mask, img_rgb = sam_segment(slice_img, slice_mask, predictor)
    combined_true_mask = (true_mask > 0)

    if slice_info[2] > 0.12 and len(label2box) > 1:
        good_slices.append(idx)

    # Calculate evaluation metrics
    dice = compute_dice(pred_mask_total, combined_true_mask)
    iou = compute_iou(pred_mask_total, combined_true_mask)
    hd = compute_hausdorff(pred_mask_total, combined_true_mask)
    dice_list_hc.append(dice)
    iou_list_hc.append(iou)
    hd_list_hc.append(hd)
    # Print processing progress
    print(f"\rProcessing slice {idx+1}/{len(mri_slices)}: {slice_label}, "
          f"Dice: {dice:.3f}, IoU: {iou:.3f}, HD: {hd:.1f}px", end='', flush=True)


In [ ]:
dice_list_hc = np.array(dice_list_hc)
iou_list_hc = np.array(iou_list_hc)
hd_list_hc = np.array([d for d in hd_list_hc if math.isfinite(d)])

# Calculate average metrics
dice_mean_hc = np.mean(dice_list_hc)
iou_mean_hc = np.mean(iou_list_hc)
hd_mean_hc = np.mean(hd_list_hc)

print(f"\n\n📊 Final Results Summary:")
print(f"   Total slices processed: {len(mri_slices)}")
print(f"   Slices with DICE < 0.5: {sum(1 for d in dice_list_hc if d < 0.5)}")
print(f"   Hippocampus MRI dataset: Average Dice = {dice_mean_hc:.4f}, Average IoU = {iou_mean_hc:.4f}, Average Hausdorff distance = {hd_mean_hc:.2f} pixel")

good_dice_mean_hc = np.mean(dice_list_hc[good_slices])
good_iou_mean_hc = np.mean(iou_list_hc[good_slices])
good_hd_mean_hc = np.mean(hd_list_hc[good_slices])
print(f"   In slices with maskedRatio > 0.12 and both labels existing:")
print(f"   Average Dice = {good_dice_mean_hc:.4f}, Average IoU = {good_iou_mean_hc:.4f}, Average Hausdorff distance = {good_hd_mean_hc:.2f} pixel")


In [ ]:
# Clean up and visualize hippocampus results
fig = None
plt.ioff()  # Turn off interactive mode
# Set up visualization
plt.ion()  # Turn on interactive mode
fig, axs = plt.subplots(3, 4)

idx = good_slices[np.random.randint(0, len(good_slices))]
slice_img, slice_mask, slice_label, slice_info = mri_slices[idx], mri_slice_masks[idx], mri_slice_labels[idx], mri_slice_summaries[idx]
label2mask, label2box, pred_mask_total, true_mask, img_rgb = sam_segment(slice_img, slice_mask, predictor)
# Visualize each structure separately
visualize_seg(fig, axs, label2box, label2mask, img_rgb, true_mask)

# Update figure title with metrics - highlight poor performance
dice, iou, hd = dice_list_hc[idx], iou_list_hc[idx], hd_list_hc[idx]
fig.suptitle(f'Slice {idx+1}/{len(mri_slices)}\n'
            f'Dice: {dice:.3f}, IoU: {iou:.3f}, HD: {hd:.1f}px',
            fontsize=14, y=0.95, color='red')
# Clean up
plt.ioff()  # Turn off interactive mode
plt.tight_layout()
plt.show()
plt.close(fig)
fig = None


In [ ]:
# Get Spleen dataset file list
ct_image_files = list(Path(f"{Task09_Spleen_dir}/imagesTr").glob("*.nii.gz"))
ct_label_files = list(Path(f"{Task09_Spleen_dir}/labelsTr").glob("*.nii.gz"))
ct_image_files = sorted([f.name for f in ct_image_files])
ct_label_files = sorted([f.name for f in ct_label_files])
print(f"Total {len(ct_image_files)} CT volumes for evaluation.")

ct_slices = []
ct_slice_masks = []

for img_file, lbl_file in zip(ct_image_files, ct_label_files):
    print(f"\rimg_file:'{img_file}', lbl_file:{lbl_file}", end='', flush=True)
    img_path = f"{Task09_Spleen_dir}/imagesTr/{img_file}"
    lbl_path = f"{Task09_Spleen_dir}/labelsTr/{lbl_file}"
    img_nii = nib.load(img_path)
    lbl_nii = nib.load(lbl_path)
    img_data = img_nii.get_fdata().astype(np.float32)
    lbl_data = lbl_nii.get_fdata().astype(np.uint8)
    # Traverse axial slices
    num_slices = img_data.shape[0]
    for k in range(num_slices):
        slice_img = img_data[k, :, :]
        slice_lbl = lbl_data[k, :, :]
        if np.any(slice_lbl == 1):  # If this slice contains spleen
            # Clip CT slice gray values to [-1000, 1000] HU range and normalize to 0-255
            slice_clip = np.clip(slice_img, -1000, 1000)
            ct_slices.append(slice_clip)
            ct_slice_masks.append(slice_lbl)  # Binary mask (0 background, 1 spleen)


In [ ]:
dice_list_spleen = []
iou_list_spleen = []
hd_list_spleen = []

for idx, (slice_img, slice_mask) in enumerate(zip(ct_slices, ct_slice_masks)):
    # if idx > 20:
    #    break
    label2mask, label2box, pred_mask_total, true_mask, img_rgb = sam_segment(slice_img, slice_mask, predictor)
    combined_true_mask = (true_mask == 1)

    # Calculate evaluation metrics
    dice = compute_dice(pred_mask_total, combined_true_mask)
    iou = compute_iou(pred_mask_total, combined_true_mask)
    hd = compute_hausdorff(pred_mask_total, combined_true_mask)
    dice_list_spleen.append(dice)
    iou_list_spleen.append(iou)
    hd_list_spleen.append(hd)
    # Print processing progress
    print(f"\rProcessing CT slice {idx+1}/{len(ct_slices)}: "
          f"Dice: {dice:.3f}, IoU: {iou:.3f}, HD: {hd:.1f}px", end='', flush=True)


In [ ]:
# Calculate average metrics for spleen
dice_mean_spleen = np.mean(dice_list_spleen)
iou_mean_spleen = np.mean(iou_list_spleen)
hd_mean_spleen = np.mean([d for d in hd_list_spleen if math.isfinite(d)])
print(f"\n\n📊 Final Results Summary:")
print(f"   Total slices processed: {len(ct_slices)}")
print(f"   Slices with DICE < 0.5: {sum(1 for d in dice_list_spleen if d < 0.5)}")
print(f"Spleen CT dataset: Average Dice = {dice_mean_spleen:.4f}, Average IoU = {iou_mean_spleen:.4f}, Average Hausdorff distance = {hd_mean_spleen:.2f} pixel")


In [ ]:
# Visualize spleen results
if fig is not None:
    fig.clear()
    plt.close(fig)
    fig = None
plt.ion()  # Turn on interactive mode
fig, axs = plt.subplots(3, 4)

idx = np.random.randint(0, len(ct_slices))
slice_img, slice_mask = ct_slices[idx], ct_slice_masks[idx]
label2mask, label2box, pred_mask_total, true_mask, img_rgb = sam_segment(slice_img, slice_mask, predictor)
visualize_seg(fig, axs, label2box, label2mask, img_rgb, true_mask)

# Update figure title with metrics - highlight poor performance
dice, iou, hd = dice_list_spleen[idx], iou_list_spleen[idx], hd_list_spleen[idx]
fig.suptitle(f'Slice {idx+1}/{len(ct_slices)}\n'
            f'Dice: {dice:.3f}, IoU: {iou:.3f}, HD: {hd:.1f}px',
            fontsize=14, y=0.95, color='red')
# Clean up
plt.ioff()  # Turn off interactive mode
plt.tight_layout()
plt.show()
plt.close(fig)
fig = None


In [ ]:
import cv2
import os
import glob

ultrasound_images = []
ultrasound_masks = []

# Process benign and malignant folders
for cls in ["benign", "malignant"]:
    image_paths = glob.glob(f"{BUSI_Dataset_dir}/{cls}/*.png")
    for img_path in image_paths:
        if "_mask" in img_path:
            continue  # Skip mask files
        # Read ultrasound image (grayscale PNG, cv2.imread reads as BGR three-channel by default)
        img_bgr = cv2.imread(img_path)
        if img_bgr is None:
            continue
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        # Create empty mask with same dimensions as image
        mask_total = np.zeros(img_rgb.shape[:2], dtype=np.uint8)
        # Image file name base part (remove path and extension)
        base_name = os.path.splitext(img_path)[0]
        # Merge all mask files for this image (may have multiple tumors)
        mask_idx = 1
        while True:
            mask_path = f"{base_name}_mask.png" if mask_idx == 1 else f"{base_name}_mask_{mask_idx}.png"
            if os.path.exists(mask_path):
                mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
                if mask is not None:
                    mask_binary = (mask > 127).astype(np.uint8)
                    mask_total = np.logical_or(mask_total, mask_binary).astype(np.uint8)
                mask_idx += 1
            else:
                break
        # If this image has tumor annotation, save it
        if mask_total.sum() > 0:
            ultrasound_images.append(img_rgb.mean(axis=2).astype(np.float32))
            ultrasound_masks.append(mask_total)

print(f"Breast ultrasound images total: {len(ultrasound_images)} (benign+malignant), masks total: {len(ultrasound_masks)}")


In [ ]:
dice_list_us = []
iou_list_us = []
hd_list_us = []

for idx, (slice_img, slice_mask) in enumerate(zip(ultrasound_images, ultrasound_masks)):
    # if idx > 20:
    #    break
    label2mask, label2box, pred_mask_total, true_mask, img_rgb = sam_segment(slice_img, slice_mask, predictor)
    combined_true_mask = (true_mask == 1)

    # Calculate evaluation metrics
    dice = compute_dice(pred_mask_total, combined_true_mask)
    iou = compute_iou(pred_mask_total, combined_true_mask)
    hd = compute_hausdorff(pred_mask_total, combined_true_mask)
    dice_list_us.append(dice)
    iou_list_us.append(iou)
    hd_list_us.append(hd)
    # Print processing progress
    print(f"\rProcessing ultrasound slice {idx+1}/{len(ultrasound_images)}: "
          f"Dice: {dice:.3f}, IoU: {iou:.3f}, HD: {hd:.1f}px", end='', flush=True)


In [ ]:
# Calculate average metrics for ultrasound
dice_mean_us = np.mean(dice_list_us)
iou_mean_us = np.mean(iou_list_us)
hd_mean_us = np.mean([d for d in hd_list_us if math.isfinite(d)])
print(f"\n\n📊 Final Results Summary:")
print(f"   Total slices processed: {len(ultrasound_images)}")
print(f"   Slices with DICE < 0.5: {sum(1 for d in dice_list_us if d < 0.5)}")
print(f"Ultrasound dataset: Average Dice = {dice_mean_us:.4f}, Average IoU = {iou_mean_us:.4f}, Average Hausdorff distance = {hd_mean_us:.2f} pixel")


In [ ]:
# Visualize ultrasound results
if fig is not None:
    fig.clear()
    plt.close(fig)
    fig = None
plt.ion()  # Turn on interactive mode
fig, axs = plt.subplots(3, 4)

idx = np.random.randint(0, len(ultrasound_images))
slice_img, slice_mask = ultrasound_images[idx], ultrasound_masks[idx]
label2mask, label2box, pred_mask_total, true_mask, img_rgb = sam_segment(slice_img, slice_mask, predictor)
visualize_seg(fig, axs, label2box, label2mask, img_rgb, true_mask)

# Update figure title with metrics - highlight poor performance
dice, iou, hd = dice_list_us[idx], iou_list_us[idx], hd_list_us[idx]
fig.suptitle(f'Slice {idx+1}/{len(ultrasound_images)}\n'
            f'Dice: {dice:.3f}, IoU: {iou:.3f}, HD: {hd:.1f}px',
            fontsize=14, y=0.95, color='red')
# Clean up
plt.ioff()  # Turn off interactive mode
plt.tight_layout()
plt.show()
plt.close(fig)
fig = None


In [ ]:
# Final comprehensive results summary
print("\n🏆 SAM Baseline Test Results Summary:")
print("=" * 60)

print(f"📊 Hippocampus MRI Dataset:")
print(f"   Mean Dice: {dice_mean_hc:.4f}")
print(f"   Mean IoU: {iou_mean_hc:.4f}")
print(f"   Mean Hausdorff: {hd_mean_hc:.2f}px")
print(f"   Total slices: {len(mri_slices)}")

print(f"\n📊 Spleen CT Dataset:")
print(f"   Mean Dice: {dice_mean_spleen:.4f}")
print(f"   Mean IoU: {iou_mean_spleen:.4f}")
print(f"   Mean Hausdorff: {hd_mean_spleen:.2f}px")
print(f"   Total slices: {len(ct_slices)}")

print(f"\n📊 Breast Ultrasound Dataset:")
print(f"   Mean Dice: {dice_mean_us:.4f}")
print(f"   Mean IoU: {iou_mean_us:.4f}")
print(f"   Mean Hausdorff: {hd_mean_us:.2f}px")
print(f"   Total images: {len(ultrasound_images)}")

print(f"\n🎯 Overall Summary:")
all_dice = dice_list_hc + dice_list_spleen + dice_list_us
all_iou = iou_list_hc + iou_list_spleen + iou_list_us
all_hd = [d for d in (hd_list_hc + hd_list_spleen + hd_list_us) if math.isfinite(d)]

print(f"   Average across all datasets:")
print(f"   Mean Dice: {np.mean(all_dice):.4f}")
print(f"   Mean IoU: {np.mean(all_iou):.4f}")
print(f"   Mean Hausdorff: {np.mean(all_hd):.2f}px")
print(f"   Total samples: {len(all_dice)}")

print(f"\n💡 Notes:")
print(f"   - This baseline uses original SAM model with bounding box prompts")
print(f"   - SAM is a general vision model, not specifically trained for medical imaging")
print(f"   - Results show capability on diverse medical imaging modalities")
print(f"   - For comparison with MedSAM which is fine-tuned for medical images")

print(f"\n✅ SAM baseline evaluation completed successfully!")
